# Etapa 1 · Obrigatório B: clima horário (Open-Meteo) para o Neon

**Campo Real Energia · Data Warehouse** · continuação do notebook `etapa1_extracao_pld.ipynb`

Rode **depois** do notebook do PLD (usa o mesmo `config.yaml`, o mesmo `.env`, o schema `externo` e a tabela `externo.log_extracao`).

O que este notebook faz:

1. Define o **ponto de clima de cada usina** (`externo.clima_ponto`): lê as coordenadas do cadastro, valida se estão dentro do Brasil e registra o critério usado. O cadastro de origem **não é alterado**.
2. Extrai o clima horário da API histórica do Open-Meteo, **uma chamada por usina** (raw: resposta JSON inteira, como chegou).
3. Reaproveita o que já foi baixado (não chama a API de novo sem necessidade) e registra tudo em `log_extracao`.
4. Trata a raw para `externo.clima_horario` (grão: **usina × hora**).
5. Verifica completude e plausibilidade.

**Atribuição obrigatória (licença de uso não comercial):** *Weather data by Open-Meteo.com*. 

In [2]:
%pip install -q psycopg2-binary pyyaml requests

Note: you may need to restart the kernel to use updated packages.


## 1. Configuração

In [3]:
import os, re, time, json, getpass
from pathlib import Path
from datetime import datetime, date, timezone

import yaml, requests, pandas as pd, psycopg2
from psycopg2.extras import execute_values, Json

CONFIG_PATH = Path("config.yaml")

BASE_YAML = """
banco:
  variavel_ambiente: NEON_DATABASE_URL
  arquivo_env: .env
http:
  timeout: 60
  max_tentativas: 5
  backoff_segundos: 2
"""

OPENMETEO_YAML = """
openmeteo:
  base_url: https://archive-api.open-meteo.com/v1/archive
  user_agent: "campo-real-dw/1.0 (trabalho academico)"
  data_inicio: "2025-01-01"    # a geracao (medicao_geracao) so existe em 2025
  data_fim: "2025-12-31"
  timezone: America/Sao_Paulo
  wind_speed_unit: ms
  pausa_entre_chamadas: 1.0
  forcar_reextracao: false     # true baixa de novo mesmo que a raw ja exista
  limites_brasil: {lat_min: -34.0, lat_max: 5.5, lon_min: -74.0, lon_max: -34.0}
  unidades_esperadas:          # se a API mudar a unidade, o tratamento para com erro
    temperature_2m: "°C"
    precipitation: "mm"
    cloud_cover: "%"
    shortwave_radiation: "W/m²"
    wind_speed_10m: "m/s"
    wind_speed_100m: "m/s"
    wind_direction_100m: "°"
"""

def mesclar(padrao, atual):
    for k, v in (atual or {}).items():
        if isinstance(v, dict) and isinstance(padrao.get(k), dict):
            mesclar(padrao[k], v)
        else:
            padrao[k] = v
    return padrao

cfg = yaml.safe_load(BASE_YAML + OPENMETEO_YAML)
if CONFIG_PATH.exists():
    atual = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}
    if "openmeteo" not in atual:
        with open(CONFIG_PATH, "a", encoding="utf-8") as f:
            f.write("\n" + OPENMETEO_YAML.strip() + "\n")
        print("Seção 'openmeteo' adicionada ao config.yaml existente.")
    cfg = mesclar(cfg, atual)
else:
    CONFIG_PATH.write_text((BASE_YAML + OPENMETEO_YAML).strip() + "\n", encoding="utf-8")
    print("config.yaml criado.")

B, H, O = cfg["banco"], cfg["http"], cfg["openmeteo"]
print("período:", O["data_inicio"], "->", O["data_fim"], "| timezone:", O["timezone"])

período: 2025-01-01 -> 2025-12-31 | timezone: America/Sao_Paulo


## 2. Conexão e log

In [4]:
def ler_url_banco():
    var = B["variavel_ambiente"]
    if os.environ.get(var):
        return os.environ[var]
    env = Path(B["arquivo_env"])
    if env.exists():
        for linha in env.read_text(encoding="utf-8").splitlines():
            linha = linha.strip()
            if linha and not linha.startswith("#") and "=" in linha:
                k, v = linha.split("=", 1)
                if k.strip() == var:
                    return v.strip().strip('"').strip("'")
    return getpass.getpass(f"Cole a connection string do Neon ({var}): ")

def conectar():
    return psycopg2.connect(ler_url_banco(), connect_timeout=30,
                            keepalives=1, keepalives_idle=30, keepalives_interval=10, keepalives_count=5)

conn = conectar()
with conn.cursor() as cur:
    cur.execute("SELECT to_regclass('externo.log_extracao')")
    if cur.fetchone()[0] is None:
        raise RuntimeError("externo.log_extracao não existe. Rode antes o notebook etapa1_extracao_pld.ipynb.")

def abrir_log(conn, recurso, etapa="EXTRACAO", fonte="OPEN_METEO"):
    with conn.cursor() as cur:
        cur.execute("INSERT INTO externo.log_extracao (etapa, fonte, recurso) VALUES (%s,%s,%s) RETURNING id_log",
                    (etapa, fonte, str(recurso)[:80]))
        id_log = cur.fetchone()[0]
    conn.commit()
    return id_log

def fechar_log(conn, id_log, lidas, gravadas, status, erro=None, obs=None):
    with conn.cursor() as cur:
        cur.execute("""UPDATE externo.log_extracao
                       SET fim=now(), linhas_lidas=%s, linhas_gravadas=%s, status=%s, erro=%s, observacao=%s
                       WHERE id_log=%s""", (lidas, gravadas, status, erro, obs, id_log))
    conn.commit()

def consulta(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

print("Conectado.")

Conectado.


## 3. Tabelas

- **`raw_openmeteo`**: uma linha por chamada (usina × período), com a **resposta JSON inteira**, as coordenadas pedidas, a URL e o momento da extração. A chave única `(cod_usina, data_inicio, data_fim)` permite reaproveitar o que já foi baixado.
- **`clima_ponto`**: **a ponte entre clima e usina**. Uma linha por usina do cadastro, com as coordenadas de origem, as coordenadas usadas e o critério. Guarda também a célula de grade que o Open-Meteo devolveu (o modelo não é um ponto exato e sim uma grade).
- **`clima_horario`** (tratada): grão **1 usina × 1 hora**, PK `(cod_usina, data_hora)`. A chave é o código de negócio da usina, porque o cadastro de origem tem um `id_usina` extra para uma duplicata.

In [5]:
DDL = """
CREATE TABLE IF NOT EXISTS externo.raw_openmeteo (
  id_raw        BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  cod_usina     VARCHAR(12)   NOT NULL,
  data_inicio   DATE          NOT NULL,
  data_fim      DATE          NOT NULL,
  latitude_req  NUMERIC(9,6)  NOT NULL,
  longitude_req NUMERIC(9,6)  NOT NULL,
  extraido_em   TIMESTAMPTZ   NOT NULL,
  origem_url    TEXT          NOT NULL,
  payload       JSONB         NOT NULL,
  UNIQUE (cod_usina, data_inicio, data_fim)
);

CREATE TABLE IF NOT EXISTS externo.clima_ponto (
  cod_usina        VARCHAR(12)  PRIMARY KEY,
  nome             VARCHAR(80),
  cod_submercado   CHAR(2),
  latitude_origem  NUMERIC(9,6),
  longitude_origem NUMERIC(9,6),
  latitude_usada   NUMERIC(9,6),
  longitude_usada  NUMERIC(9,6),
  criterio         VARCHAR(30)  NOT NULL,   -- origem | sinal_latitude_corrigido | excluida
  observacao       TEXT,
  lat_grade        NUMERIC(9,6),            -- célula de grade devolvida pela API
  lon_grade        NUMERIC(9,6),
  elevacao_m       NUMERIC(7,1),
  atualizado_em    TIMESTAMPTZ  NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS externo.clima_horario (
  cod_usina            VARCHAR(12)   NOT NULL,
  data_hora            TIMESTAMP     NOT NULL,       -- início da hora, horário de Brasília
  temperatura_c        NUMERIC(5,2),
  precipitacao_mm      NUMERIC(6,2),
  cobertura_nuvens_pct SMALLINT,
  radiacao_w_m2        NUMERIC(7,2),                 -- radiação de onda curta (global horizontal)
  vento_10m_ms         NUMERIC(5,2),
  vento_100m_ms        NUMERIC(5,2),
  dir_vento_100m_graus SMALLINT,
  id_raw               BIGINT        NOT NULL,
  extraido_em          TIMESTAMPTZ   NOT NULL,
  carregado_em         TIMESTAMPTZ   NOT NULL DEFAULT now(),
  PRIMARY KEY (cod_usina, data_hora)
);
"""
with conn.cursor() as cur:
    cur.execute(DDL)
conn.commit()
print("Tabelas prontas.")

Tabelas prontas.


## 4. Ponto de clima de cada usina

Regras, aplicadas na ordem (o cadastro de origem **não** é tocado):

1. `cod_usina` fora do padrão `CRE-XXX-000` → **excluída**. É o caso da duplicata de cadastro `CRE-EOL-07`, que repete as coordenadas de `CRE-EOL-007` e geraria clima duplicado.
2. Coordenada dentro do retângulo do Brasil → usada **como está** (`origem`).
3. Coordenada fora do Brasil, mas que passa a valer ao **inverter o sinal da latitude** → `sinal_latitude_corrigido`. Justificativa: o ponto original cai no oceano, o município cadastrado fica no Brasil, e só o sinal difere. A correção fica registrada em `criterio` e `observacao`.
4. Qualquer outro caso → **excluída** (não se inventa coordenada).

In [6]:
L = O["limites_brasil"]
def no_brasil(lat, lon):
    return L["lat_min"] <= lat <= L["lat_max"] and L["lon_min"] <= lon <= L["lon_max"]

with conn.cursor() as cur:
    cur.execute("""SELECT cod_usina, nome, cod_submercado, municipio, uf, latitude, longitude
                   FROM campo_real_energia.usina ORDER BY id_usina""")
    usinas = cur.fetchall()

pontos = []
for cod, nome, sub, mun, uf, lat, lon in usinas:
    lat, lon = float(lat), float(lon)
    if not re.fullmatch(r"CRE-[A-Z]{3}-\d{3}", cod):
        crit, la, lo, obs = "excluida", None, None, "cod_usina fora do padrão (duplicata de cadastro); mesmas coordenadas de outra usina"
    elif no_brasil(lat, lon):
        crit, la, lo, obs = "origem", lat, lon, None
    elif no_brasil(-lat, lon):
        crit, la, lo = "sinal_latitude_corrigido", -lat, lon
        obs = f"latitude {lat} fora do Brasil; sinal invertido para {-lat} ({mun}/{uf})"
    else:
        crit, la, lo, obs = "excluida", None, None, f"coordenada ({lat}, {lon}) fora do Brasil e sem correção segura"
    pontos.append((cod, nome, sub, lat, lon, la, lo, crit, obs))

with conn.cursor() as cur:
    execute_values(cur, """
        INSERT INTO externo.clima_ponto
          (cod_usina, nome, cod_submercado, latitude_origem, longitude_origem,
           latitude_usada, longitude_usada, criterio, observacao)
        VALUES %s
        ON CONFLICT (cod_usina) DO UPDATE SET
          nome=EXCLUDED.nome, cod_submercado=EXCLUDED.cod_submercado,
          latitude_origem=EXCLUDED.latitude_origem, longitude_origem=EXCLUDED.longitude_origem,
          latitude_usada=EXCLUDED.latitude_usada, longitude_usada=EXCLUDED.longitude_usada,
          criterio=EXCLUDED.criterio, observacao=EXCLUDED.observacao, atualizado_em=now()
    """, pontos)
conn.commit()

display(consulta("""SELECT cod_usina, cod_submercado, latitude_origem, longitude_origem,
                           latitude_usada, longitude_usada, criterio, observacao
                    FROM externo.clima_ponto ORDER BY cod_usina"""))

,cod_usina,cod_submercado,latitude_origem,longitude_origem,latitude_usada,longitude_usada,criterio,observacao
0,CRE-BIO-005,SE,-21.138000,-47.990000,-21.138000,-47.990000,origem,NaN
1,CRE-EOL-003,S,-30.890000,-55.530000,-30.890000,-55.530000,origem,NaN
2,CRE-EOL-007,NE,-5.537000,-35.819000,-5.537000,-35.819000,origem,NaN
3,CRE-EOL-07,NE,-5.537000,-35.819000,None,None,excluida,cod_usina fora do padrão (duplicata de cadastr...
4,CRE-PCH-002,S,-26.775000,-51.015000,-26.775000,-51.015000,origem,NaN
5,CRE-UFV-004,SE,15.803000,-43.308000,-15.803000,-43.308000,sinal_latitude_corrigido,latitude 15.803 fora do Brasil; sinal invertid...
6,CRE-UFV-006,NE,-13.255000,-43.418000,-13.255000,-43.418000,origem,NaN
7,CRE-UHE-001,S,-25.480000,-53.610000,-25.480000,-53.610000,origem,NaN
8,CRE-UHE-008,N,-12.025000,-48.540000,-12.025000,-48.540000,origem,NaN
9,CRE-UTE-009,N,-4.867000,-44.360000,-4.867000,-44.360000,origem,NaN


## 5. Extração (camada raw)

Cada usina é **uma chamada** com o ano inteiro. Tratamentos de falha: timeout e erro de conexão (retry com backoff exponencial), 429 e 5xx (espera, respeitando `Retry-After` quando existe), resposta que não é JSON, erro explícito da API (`error: true`, sem retry), resposta vazia, variável ausente e séries de tamanhos diferentes. Se a raw da usina e do período já existe, a chamada **não é feita** (status `PULADO` no log), a menos que `forcar_reextracao` seja `true`.

O vento vem em m/s (`wind_speed_unit`), a hora em horário de Brasília (`timezone`).

In [7]:
VARIAVEIS = {   # variável da API -> coluna da tabela tratada (ordem = ordem do INSERT)
    "temperature_2m":      "temperatura_c",
    "precipitation":       "precipitacao_mm",
    "cloud_cover":         "cobertura_nuvens_pct",
    "shortwave_radiation": "radiacao_w_m2",
    "wind_speed_10m":      "vento_10m_ms",
    "wind_speed_100m":     "vento_100m_ms",
    "wind_direction_100m": "dir_vento_100m_graus",
}

def _espera(headers, padrao):
    try:
        return int(headers.get("Retry-After", padrao))
    except (TypeError, ValueError):
        return padrao

def http_json(url, params):
    ultimo = None
    for t in range(1, H["max_tentativas"] + 1):
        padrao = H["backoff_segundos"] * 2 ** (t - 1)
        try:
            r = requests.get(url, params=params, timeout=H["timeout"], headers={"User-Agent": O["user_agent"]})
        except requests.RequestException as e:
            ultimo = f"{type(e).__name__}: {e}"
            time.sleep(padrao)
            continue
        if r.status_code == 429 or r.status_code >= 500:
            ultimo = f"HTTP {r.status_code}: {r.text[:150]}"
            time.sleep(max(_espera(r.headers, 0), 30 if r.status_code == 429 else padrao))
            continue
        try:
            j = r.json()
        except ValueError:
            ultimo = f"resposta não-JSON: {r.text[:120]!r}"
            time.sleep(padrao)
            continue
        if r.status_code != 200 or (isinstance(j, dict) and j.get("error")):
            motivo = j.get("reason") if isinstance(j, dict) else j
            raise ValueError(f"Open-Meteo recusou (HTTP {r.status_code}): {motivo}")
        return j, r.url
    raise RuntimeError(f"Falha após {H['max_tentativas']} tentativas. Último erro: {ultimo}")

def extrair_ponto(conn, cod, lat, lon):
    recurso = f"{cod} {O['data_inicio']}..{O['data_fim']}"
    id_log = abrir_log(conn, recurso)
    try:
        if not O["forcar_reextracao"]:
            with conn.cursor() as cur:
                cur.execute("""SELECT 1 FROM externo.raw_openmeteo
                               WHERE cod_usina=%s AND data_inicio=%s AND data_fim=%s""",
                            (cod, O["data_inicio"], O["data_fim"]))
                if cur.fetchone():
                    fechar_log(conn, id_log, 0, 0, "PULADO", obs="raw já existe; nenhuma chamada à API")
                    return "pulado"
        params = {"latitude": lat, "longitude": lon,
                  "start_date": O["data_inicio"], "end_date": O["data_fim"],
                  "hourly": ",".join(VARIAVEIS), "timezone": O["timezone"],
                  "wind_speed_unit": O["wind_speed_unit"]}
        dados, url = http_json(O["base_url"], params)
        h = dados.get("hourly") or {}
        tempos = h.get("time") or []
        if not tempos:
            raise ValueError("resposta vazia: sem horas")
        faltam = [v for v in VARIAVEIS if v not in h]
        if faltam:
            raise ValueError(f"variáveis ausentes na resposta: {faltam}")
        if any(len(h[v]) != len(tempos) for v in VARIAVEIS):
            raise ValueError("séries com tamanhos diferentes")
        with conn.cursor() as cur:
            cur.execute("""
                INSERT INTO externo.raw_openmeteo
                  (cod_usina, data_inicio, data_fim, latitude_req, longitude_req, extraido_em, origem_url, payload)
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
                ON CONFLICT (cod_usina, data_inicio, data_fim) DO UPDATE SET
                  latitude_req=EXCLUDED.latitude_req, longitude_req=EXCLUDED.longitude_req,
                  extraido_em=EXCLUDED.extraido_em, origem_url=EXCLUDED.origem_url, payload=EXCLUDED.payload
            """, (cod, O["data_inicio"], O["data_fim"], lat, lon, datetime.now(timezone.utc), url, Json(dados)))
        conn.commit()
        fechar_log(conn, id_log, len(tempos), len(tempos), "SUCESSO")
        return "ok"
    except Exception as e:
        conn.rollback()
        fechar_log(conn, id_log, 0, 0, "ERRO", erro=str(e)[:500])
        raise

with conn.cursor() as cur:
    cur.execute("""SELECT cod_usina, latitude_usada, longitude_usada FROM externo.clima_ponto
                   WHERE criterio <> 'excluida' ORDER BY cod_usina""")
    alvos = [(c, float(a), float(b)) for c, a, b in cur.fetchall()]

resultado = {}
for cod, lat, lon in alvos:
    try:
        r = extrair_ponto(conn, cod, lat, lon)
        resultado[cod] = r
        print(f"{cod}: {r}")
        if r == "ok":
            time.sleep(O["pausa_entre_chamadas"])
    except Exception as e:
        resultado[cod] = "ERRO"
        print(f"{cod}: ERRO -> {e}")
print("\nresumo:", pd.Series(resultado).value_counts().to_dict())

CRE-BIO-005: pulado
CRE-EOL-003: pulado
CRE-EOL-007: pulado
CRE-PCH-002: pulado
CRE-UFV-004: pulado
CRE-UFV-006: pulado
CRE-UHE-001: pulado
CRE-UHE-008: pulado
CRE-UTE-009: pulado

resumo: {'pulado': 9}


## 6. Tratamento (raw → `externo.clima_horario`)

- confere as **unidades** devolvidas pela API contra `unidades_esperadas` (se mudaram, para com erro);
- converte o texto de hora em `timestamp` (já vem em horário de Brasília por causa do parâmetro `timezone`);
- **valores nulos da API ficam nulos**: não se imputa clima; a quantidade vai para a coluna `observacao` do log;
- confere o número de horas contra o esperado para o período;
- grava a célula de grade e a elevação devolvidas pela API em `clima_ponto`.

In [8]:
COLS = list(VARIAVEIS.values())

def tratar_raw(conn, id_raw, cod, ini, fim, extraido_em, payload):
    id_log = abrir_log(conn, f"{cod} {ini}..{fim}", "TRATAMENTO")
    lidas = gravadas = 0
    try:
        h, unidades = payload["hourly"], payload.get("hourly_units", {})
        divergentes = {v: (unidades.get(v), esp) for v, esp in O["unidades_esperadas"].items()
                       if unidades.get(v) != esp}
        if divergentes:
            raise ValueError(f"unidade mudou (recebida, esperada): {divergentes}")
        tempos = [datetime.fromisoformat(t) for t in h["time"]]
        lidas = len(tempos)
        esperadas = ((fim - ini).days + 1) * 24
        rows = [(cod, t, *[h[v][i] for v in VARIAVEIS], id_raw, extraido_em)
                for i, t in enumerate(tempos)]
        nulos = {c: sum(1 for x in h[v] if x is None) for v, c in VARIAVEIS.items()}
        nulos = {c: n for c, n in nulos.items() if n}
        with conn.cursor() as cur:
            execute_values(cur, f"""
                INSERT INTO externo.clima_horario
                  (cod_usina, data_hora, {", ".join(COLS)}, id_raw, extraido_em)
                VALUES %s
                ON CONFLICT (cod_usina, data_hora) DO UPDATE SET
                  {", ".join(f"{c}=EXCLUDED.{c}" for c in COLS)},
                  id_raw=EXCLUDED.id_raw, extraido_em=EXCLUDED.extraido_em, carregado_em=now()
            """, rows, page_size=1000)
            cur.execute("""UPDATE externo.clima_ponto
                           SET lat_grade=%s, lon_grade=%s, elevacao_m=%s, atualizado_em=now()
                           WHERE cod_usina=%s""",
                        (payload.get("latitude"), payload.get("longitude"), payload.get("elevation"), cod))
        conn.commit()
        gravadas = len(rows)
        avisos = []
        if lidas != esperadas:
            avisos.append(f"horas={lidas} (esperado {esperadas})")
        if nulos:
            avisos.append(f"nulos={nulos}")
        fechar_log(conn, id_log, lidas, gravadas, "SUCESSO", obs="; ".join(avisos) or None)
        print(f"  {cod}: lidas={lidas} gravadas={gravadas} {'; '.join(avisos)}")
    except Exception as e:
        conn.rollback()
        fechar_log(conn, id_log, lidas, 0, "ERRO", erro=str(e)[:500])
        raise

with conn.cursor() as cur:
    cur.execute("""SELECT id_raw, cod_usina, data_inicio, data_fim, extraido_em, payload
                   FROM externo.raw_openmeteo ORDER BY cod_usina""")
    raws = cur.fetchall()

for id_raw, cod, ini, fim, ext, payload in raws:
    try:
        tratar_raw(conn, id_raw, cod, ini, fim, ext, payload)
    except Exception as e:
        print(f"  {cod}: ERRO -> {e}")

  CRE-BIO-005: lidas=8760 gravadas=8760 
  CRE-EOL-003: lidas=8760 gravadas=8760 
  CRE-EOL-007: lidas=8760 gravadas=8760 
  CRE-PCH-002: lidas=8760 gravadas=8760 
  CRE-UFV-004: lidas=8760 gravadas=8760 
  CRE-UFV-006: lidas=8760 gravadas=8760 
  CRE-UHE-001: lidas=8760 gravadas=8760 
  CRE-UHE-008: lidas=8760 gravadas=8760 
  CRE-UTE-009: lidas=8760 gravadas=8760 


## 7. Verificação

In [9]:
display(consulta("""
    SELECT cod_usina,
           COUNT(*)                                   AS horas,
           MIN(data_hora)                             AS primeira,
           MAX(data_hora)                             AS ultima,
           COUNT(*) FILTER (WHERE temperatura_c IS NULL)  AS nulos_temp,
           COUNT(*) FILTER (WHERE radiacao_w_m2 IS NULL)  AS nulos_rad,
           COUNT(*) FILTER (WHERE vento_100m_ms IS NULL)  AS nulos_vento,
           ROUND(AVG(radiacao_w_m2)::numeric, 1)      AS rad_media_w_m2,
           ROUND(MAX(radiacao_w_m2)::numeric, 0)      AS rad_max_w_m2,
           ROUND(AVG(vento_100m_ms)::numeric, 2)      AS vento100_medio_ms,
           ROUND(AVG(temperatura_c)::numeric, 1)      AS temp_media_c,
           ROUND(SUM(precipitacao_mm)::numeric, 0)    AS chuva_total_mm
    FROM externo.clima_horario
    GROUP BY cod_usina ORDER BY cod_usina
"""))
print("Referência: ano de 2025 = 8760 horas por usina; 8 usinas esperadas se a duplicata foi excluída (9 no cadastro válido).")

,cod_usina,horas,primeira,ultima,nulos_temp,nulos_rad,nulos_vento,rad_media_w_m2,rad_max_w_m2,vento100_medio_ms,temp_media_c,chuva_total_mm
0,CRE-BIO-005,8760,2025-01-01,2025-12-31 23:00:00,0,0,0,233.0,1100,4.47,23.6,986
1,CRE-EOL-003,8760,2025-01-01,2025-12-31 23:00:00,0,0,0,207.6,1101,5.10,18.0,1493
2,CRE-EOL-007,8760,2025-01-01,2025-12-31 23:00:00,0,0,0,243.3,1047,7.44,26.0,724
3,CRE-PCH-002,8760,2025-01-01,2025-12-31 23:00:00,0,0,0,205.8,1127,4.75,17.0,1538
4,CRE-UFV-004,8760,2025-01-01,2025-12-31 23:00:00,0,0,0,249.5,1087,5.07,25.1,625
5,CRE-UFV-006,8760,2025-01-01,2025-12-31 23:00:00,0,0,0,257.5,1080,4.01,26.6,634
6,CRE-UHE-001,8760,2025-01-01,2025-12-31 23:00:00,0,0,0,210.2,1098,4.31,21.5,1914
7,CRE-UHE-008,8760,2025-01-01,2025-12-31 23:00:00,0,0,0,241.5,1028,4.21,27.4,1114
8,CRE-UTE-009,8760,2025-01-01,2025-12-31 23:00:00,0,0,0,237.5,1005,3.57,27.1,1659


Referência: ano de 2025 = 8760 horas por usina; 8 usinas esperadas se a duplicata foi excluída (9 no cadastro válido).


In [10]:
print("Ponto pedido x célula de grade devolvida pela API:")
display(consulta("""SELECT cod_usina, criterio, latitude_usada, longitude_usada, lat_grade, lon_grade, elevacao_m
                    FROM externo.clima_ponto ORDER BY cod_usina"""))

print("\nÚltimas execuções:")
display(consulta("""SELECT id_log, etapa, fonte, recurso, linhas_lidas, linhas_gravadas, status, erro, observacao
                    FROM externo.log_extracao WHERE fonte='OPEN_METEO' ORDER BY id_log DESC LIMIT 30"""))

Ponto pedido x célula de grade devolvida pela API:


,cod_usina,criterio,latitude_usada,longitude_usada,lat_grade,lon_grade,elevacao_m
0,CRE-BIO-005,origem,-21.138000,-47.990000,-21.124780,-48.018280,546.0
1,CRE-EOL-003,origem,-30.890000,-55.530000,-30.896309,-55.491120,207.0
2,CRE-EOL-007,origem,-5.537000,-35.819000,-5.518453,-35.820892,156.0
3,CRE-EOL-07,excluida,None,None,None,None,None
4,CRE-PCH-002,origem,-26.775000,-51.015000,-26.818981,-51.029907,911.0
5,CRE-UFV-004,sinal_latitude_corrigido,-15.803000,-43.308000,-15.782073,-43.301880,535.0
6,CRE-UFV-006,origem,-13.255000,-43.418000,-13.251318,-43.439790,435.0
7,CRE-UHE-001,origem,-25.480000,-53.610000,-25.483305,-53.590027,311.0
8,CRE-UHE-008,origem,-12.025000,-48.540000,-12.056238,-48.517517,243.0
9,CRE-UTE-009,origem,-4.867000,-44.360000,-4.885764,-44.370360,117.0



Últimas execuções:


,id_log,etapa,fonte,recurso,linhas_lidas,linhas_gravadas,status,erro,observacao
0,41,TRATAMENTO,OPEN_METEO,CRE-UTE-009 2025-01-01..2025-12-31,8760,8760,SUCESSO,None,NaN
1,40,TRATAMENTO,OPEN_METEO,CRE-UHE-008 2025-01-01..2025-12-31,8760,8760,SUCESSO,None,NaN
2,39,TRATAMENTO,OPEN_METEO,CRE-UHE-001 2025-01-01..2025-12-31,8760,8760,SUCESSO,None,NaN
3,38,TRATAMENTO,OPEN_METEO,CRE-UFV-006 2025-01-01..2025-12-31,8760,8760,SUCESSO,None,NaN
4,37,TRATAMENTO,OPEN_METEO,CRE-UFV-004 2025-01-01..2025-12-31,8760,8760,SUCESSO,None,NaN
5,36,TRATAMENTO,OPEN_METEO,CRE-PCH-002 2025-01-01..2025-12-31,8760,8760,SUCESSO,None,NaN
6,35,TRATAMENTO,OPEN_METEO,CRE-EOL-007 2025-01-01..2025-12-31,8760,8760,SUCESSO,None,NaN
7,34,TRATAMENTO,OPEN_METEO,CRE-EOL-003 2025-01-01..2025-12-31,8760,8760,SUCESSO,None,NaN
8,33,TRATAMENTO,OPEN_METEO,CRE-BIO-005 2025-01-01..2025-12-31,8760,8760,SUCESSO,None,NaN
9,32,EXTRACAO,OPEN_METEO,CRE-UTE-009 2025-01-01..2025-12-31,0,0,PULADO,None,raw já existe; nenhuma chamada à API
